# Create train/val/test splits from meta

Splits are derived from **data/benchmark** (Parquet + meta.json) using `eyefeatures.data`:

- **Data**: `eyefeatures.data.load_dataset` (Parquet)
- **Split groups**: from `meta.json` per-dataset `labels[label_col].splitting_column`
- **Output**: `splits_dir/{dataset}_labels.csv`, `{dataset}_{label}_split_info.json` per label

Run this once before feature extraction notebooks.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from utils.benchmark_utils import (
    get_benchmark_dir,
    list_datasets,
    load_dataset_with_meta,
    create_and_save_splits_for_dataset,
)

OUTPUT_DIR = Path('features_output')
OUTPUT_DIR.mkdir(exist_ok=True)
SPLITS_DIR = OUTPUT_DIR / 'splits'
SPLITS_DIR.mkdir(exist_ok=True)

BENCHMARK_DIR = get_benchmark_dir()

In [ ]:
# List fixation datasets (Parquet)
dataset_names = list_datasets(dataset_type='fixation')
print(f"Found {len(dataset_names)} fixation datasets")

In [ ]:
for name in dataset_names:
    try:
        df, meta_info = load_dataset_with_meta(name)
        if meta_info.get('info') and meta_info['info'].get('labels'):
            labels_path, split_paths = create_and_save_splits_for_dataset(
                name, df, meta_info, SPLITS_DIR,
                test_size=0.2, val_size=0.2, random_state=42, overwrite=False
            )
            print(f"{name}: {len(split_paths)} label splits")
        else:
            print(f"{name}: no meta labels, skip")
    except Exception as e:
        print(f"{name}: ERROR {e}")